# Week 6 — Advanced Data Analysis

# Objective
The objective of Week 6 is to build on the findings from Week 5 by conducting deeper analytical and segment-level analysis of appointment attendance and no-show behaviour.
Rather than repeating the exploratory analysis already completed in Week 4 and Week 5, this analysis will investigate relationships between key variables to identify higher-risk appointment and patient segments.
The analysis will focus on:
* Booking Lead Time
* Previous No-Shows
* Reminder Status
* Distance to Clinic
* Waiting Time
  
The findings will be used to validate and refine the conclusions from Week 5, develop more targeted business recommendations, and identify useful patterns and variables that can support the predictive modelling stage of the HealthConnect Clinic project.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("HealthConnect_Appointment_Data_Cleaned.csv")

## Dataset Readiness Check
The Week 5 cleaned dataset is used as the starting point for the Week 6 analysis.
Before conducting the advanced analysis, the dataset will be checked to confirm its structure, dimensions, data types, and overall readiness for further analysis.

In [3]:
print("Dataset Shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

Dataset Shape: (5000, 19)

Column names:
['appointment_id', 'patient_id', 'gender', 'age', 'age_group', 'appointment_type', 'booking_date', 'appointment_date', 'appointment_day', 'appointment_time', 'booking_lead_days', 'previous_appointments', 'previous_no_shows', 'reminder_sent', 'reminder_channel', 'distance_to_clinic_km', 'waiting_time_minutes', 'appointment_outcome', 'lead_time_group']

Data types:
appointment_id            object
patient_id                object
gender                    object
age                        int64
age_group                 object
appointment_type          object
booking_date              object
appointment_date          object
appointment_day           object
appointment_time          object
booking_lead_days          int64
previous_appointments      int64
previous_no_shows          int64
reminder_sent             object
reminder_channel          object
distance_to_clinic_km    float64
waiting_time_minutes     float64
appointment_outcome       object

In [4]:
df["booking_date"] = pd.to_datetime(df["booking_date"])
df["appointment_date"] = pd.to_datetime(df["appointment_date"])

In [5]:
df[["booking_date", "appointment_date"]].dtypes

booking_date        datetime64[ns]
appointment_date    datetime64[ns]
dtype: object

In [6]:
outcome_summary = df["appointment_outcome"].value_counts().to_frame("Count")
outcome_summary["Percentage"] = (df["appointment_outcome"].value_counts(normalize=True) * 100).round(2)

print(outcome_summary)

                     Count  Percentage
appointment_outcome                   
No-Show               2423       48.46
Attended              2314       46.28
Cancelled              263        5.26


## Calculate Week 5 KPIs
The key performance indicators from Week 5 are recalculated using the current dataset to confirm that the baseline results remain consistent.
The KPIs include the total appointment, appointment no-show rate, attendance rate, reminder coverage rate, and average booking lead time.

In [7]:
total_appointments = len(df)

no_show_rate = ((df["appointment_outcome"] == "No-Show").sum() / total_appointments) * 100

attendance_rate = ((df["appointment_outcome"] == "Attended").sum() / total_appointments) * 100

reminder_coverage_rate = ((df["reminder_sent"] == "Yes").sum() / total_appointments) * 100

average_booking_lead_time = df["booking_lead_days"].mean()

print(f"Total Appointment:{total_appointments:.2f}")
print(f"No-Show Rate: {no_show_rate:.2f}%")
print(f"Attendance Rate: {attendance_rate:.2f}%")
print(f"Reminder Coverage Rate: {reminder_coverage_rate:.2f}%")
print(f"Average Booking Lead Time: {average_booking_lead_time:.2f} days")

Total Appointment:5000.00
No-Show Rate: 48.46%
Attendance Rate: 46.28%
Reminder Coverage Rate: 72.68%
Average Booking Lead Time: 29.64 days


## 3. Advanced Analysis 1 — Booking Lead Time × Previous No-Shows
Week 5 showed that longer booking lead times were associated with higher no-show rates, while patients with previous no-shows also appeared more likely to miss their current appointments.
This analysis examines these two factors together to determine whether the combination of longer booking lead times and previous no-show history identifies specific patient or appointment segments with particularly high no-show rates.
The results will help identify higher-risk segments that may be useful for targeted clinic interventions and future predictive modelling.

### 3.1 Review Existing Booking Lead-Time Groups
The booking lead-time categories created during the earlier analysis are reviewed and used for the Week 6 advanced analysis.
Using the existing categories ensures consistency with the Week 5 findings and avoids repeating data preparation that has already been completed.

In [8]:
df["lead_time_group"].value_counts(dropna=False)

lead_time_group
15-30 days    1333
31-45 days    1258
46-60 days    1167
0-7 days       640
8-14 days      602
Name: count, dtype: int64

#### 3.1.1 Order Existing Booking Lead-Time Groups
The existing booking lead-time groups are arranged in chronological order to ensure that subsequent analysis presents the categories from the shortest to the longest booking lead time.
No new groups are created; only the order of the existing categories is adjusted.

In [9]:
lead_time_order = [
    "0-7 days",
    "8-14 days",
    "15-30 days",
    "31-45 days",
    "46-60 days"
]
lead_time_counts = df["lead_time_group"].value_counts()
lead_time_counts = lead_time_counts.reindex(lead_time_order)  
lead_time_counts

lead_time_group
0-7 days       640
8-14 days      602
15-30 days    1333
31-45 days    1258
46-60 days    1167
Name: count, dtype: int64

### 3.2 Review Previous No-Show History
The previous_no_shows variable records the number of previous appointments that each patient failed to attend.
This variable was examined during the earlier analysis. It will now be reviewed as part of the Week 6 advanced analysis before being combined with booking lead-time groups.
The purpose is to understand the distribution of previous no-show history and establish the categories that will be used to examine the interaction between previous no-shows and booking lead time.

In [10]:
df["previous_no_shows"].value_counts().sort_index()

previous_no_shows
0    2921
1    1548
2     438
3      78
4      12
5       3
Name: count, dtype: int64

#### 3.2.1 Group Previous No-Show History
The distribution shows that most patients had either no previous no-shows or one previous no-show. The number of patients decreases as the number of previous no-shows increases, with only a small number of patients having two or more previous no-shows.
For the Week 6 analysis, previous no-show history will be grouped into three categories:
* 0 — No previous no-shows
* 1 — One previous no-show
* 2+ — Two or more previous no-shows
  
This grouping provides a clearer basis for comparing previous no-show history with booking lead time.

In [11]:
df["previous_no_show_group"] = df["previous_no_shows"].apply(
    lambda x: "0" if x == 0 else "1" if x == 1 else "2+"
)
df["previous_no_show_group"].value_counts()

previous_no_show_group
0     2921
1     1548
2+     531
Name: count, dtype: int64

### 3.3 Booking Lead Time × Previous No-Show History
This analysis combines booking lead time with previous no-show history to determine whether no-show rates vary across these two factors.
The aim is to identify appointment segments with higher no-show rates and assess whether the patterns observed individually in Week 5 remain evident when the variables are analysed together.

In [12]:
segment_analysis = (
    df.groupby(["lead_time_group", "previous_no_show_group"])
      .agg(
          appointments=("appointment_id", "count"),
          no_shows=("appointment_outcome", lambda x: (x == "No-Show").sum())
      )
      .reset_index()
)

segment_analysis["no_show_rate"] = (
    segment_analysis["no_shows"] /
    segment_analysis["appointments"] * 100
)

segment_analysis

,lead_time_group,previous_no_show_group,appointments,no_shows,no_show_rate
0,0-7 days,0,372,81,21.774194
1,0-7 days,1,190,63,33.157895
2,0-7 days,2+,78,34,43.589744
3,15-30 days,0,763,293,38.401048
4,15-30 days,1,419,196,46.778043
5,15-30 days,2+,151,87,57.615894
6,31-45 days,0,742,366,49.326146
7,31-45 days,1,406,242,59.605911
8,31-45 days,2+,110,69,62.727273
9,46-60 days,0,668,412,61.676647


#### 3.3.1 Order Booking Lead-Time Groups
The booking lead-time groups are arranged from the shortest to the longest lead time to make comparisons across previous no show easier to interpret.

In [13]:
lead_time_order = [
    "0-7 days",
    "8-14 days",
    "15-30 days",
    "31-45 days",
    "46-60 days"
]

segment_analysis["lead_time_group"] = pd.Categorical(
    segment_analysis["lead_time_group"],
    categories=lead_time_order,
    ordered=True
)

segment_analysis = segment_analysis.sort_values(
    ["lead_time_group", "previous_no_show_group"]
).reset_index(drop=True)

segment_analysis

,lead_time_group,previous_no_show_group,appointments,no_shows,no_show_rate
0,0-7 days,0,372,81,21.774194
1,0-7 days,1,190,63,33.157895
2,0-7 days,2+,78,34,43.589744
3,8-14 days,0,376,119,31.648936
4,8-14 days,1,157,50,31.847134
5,8-14 days,2+,69,33,47.826087
6,15-30 days,0,763,293,38.401048
7,15-30 days,1,419,196,46.778043
8,15-30 days,2+,151,87,57.615894
9,31-45 days,0,742,366,49.326146


### 3.4 Review Segment Appointment Counts
Before interpreting the no-show rates, the number of appointments within each booking lead-time and previous no-show segment will be reviewed.
This helps determine whether the observed no-show rates are based on sufficiently large groups and provides context for interpreting the results.

In [14]:
segment_counts = pd.crosstab(
    df["lead_time_group"],
    df["previous_no_show_group"]
)
segment_counts = segment_counts.reindex(lead_time_order)
segment_counts

previous_no_show_group,0,1,2+
lead_time_group,,,
0-7 days,372,190,78
8-14 days,376,157,69
15-30 days,763,419,151
31-45 days,742,406,110
46-60 days,668,376,123


### 3.5 Compare No-Show Rates
The no-show rate for appoinments with different no-show histories will be compared across the booking lead-time groups. This comparison will show whether previous no-show history and longer booking lead times are associated with higher no-show rate

In [15]:
lead_time_order = [
    "0-7 days",
    "8-14 days",
    "15-30 days",
    "31-45 days",
    "46-60 days"
]

lead_time_no_show = pd.crosstab(
    df["lead_time_group"],
    df["previous_no_show_group"],
    values=df["appointment_outcome"].eq("No-Show"),
    aggfunc="mean"
) * 100

lead_time_no_show = lead_time_no_show.reindex(lead_time_order)  
lead_time_no_show.round(2)

previous_no_show_group,0,1,2+
lead_time_group,,,
0-7 days,21.77,33.16,43.59
8-14 days,31.65,31.85,47.83
15-30 days,38.40,46.78,57.62
31-45 days,49.33,59.61,62.73
46-60 days,61.68,73.67,82.11


### 3.6 Identify High- and Low-Risk Segments
The combined no-show rates will be reviewed to identify the segments with the highest and lowest rates.
This will help determine whether patients with previous no-shows are more likely to miss appointments at longer booking lead times and highlight segments that may require greater attention.

In [16]:
highest = segment_analysis.loc[
    segment_analysis["no_show_rate"].idxmax()
]

lowest = segment_analysis.loc[
    segment_analysis["no_show_rate"].idxmin()
]

print(f"Highest no-show rate: {highest['no_show_rate']:.2f}%")
print(
    f"Segment: {highest['lead_time_group']} "
    f"and {highest['previous_no_show_group']} previous no-shows"
)
print(f"Appointments: {highest['appointments']}")

print(f"\nLowest no-show rate: {lowest['no_show_rate']:.2f}%")
print(
    f"Segment: {lowest['lead_time_group']} "
    f"and {lowest['previous_no_show_group']} previous no-shows"
)
print(f"Appointments: {lowest['appointments']}")

Highest no-show rate: 82.11%
Segment: 46-60 days and 2+ previous no-shows
Appointments: 123

Lowest no-show rate: 21.77%
Segment: 0-7 days and 0 previous no-shows
Appointments: 372


In [17]:
lead_time_no_show.round(2)

previous_no_show_group,0,1,2+
lead_time_group,,,
0-7 days,21.77,33.16,43.59
8-14 days,31.65,31.85,47.83
15-30 days,38.40,46.78,57.62
31-45 days,49.33,59.61,62.73
46-60 days,61.68,73.67,82.11


The results show that no-show rates generally increase as booking lead time increases. Within most lead-time groups, patients with previous no-shows also have higher no-show rates than those with no previous no-shows.
The highest no-show rate was observed among patients with 2+ previous no-shows and a 46–60 day booking lead time (82.11%). The lowest was among patients with no previous no-shows and a 0–7 day lead time (21.77%).
Overall, the combination of longer booking lead time and previous no-show history identifies appointment segments with substantially higher no-show rates.

## 4. Advanced Analysis 2 — Booking Lead Time × Reminder Status
Week 5 showed a difference in no-show rates between appointments where a reminder was sent and those where no reminder was sent.
This analysis combines reminder status with booking lead time to determine whether the relationship between lead time and no-shows differs depending on whether a reminder was sent.
The aim is to identify whether longer booking lead times are associated with higher no-show rates within both reminder groups.

In [18]:
lead_time_reminder = (
    df.groupby(["lead_time_group", "reminder_sent"])
      .agg(
          appointments=("appointment_id", "count"),
          no_shows=("appointment_outcome", lambda x: (x == "No-Show").sum())
      )
      .reset_index()
)

lead_time_reminder["no_show_rate"] = (
    lead_time_reminder["no_shows"] /
    lead_time_reminder["appointments"] * 100
)

lead_time_reminder

,lead_time_group,reminder_sent,appointments,no_shows,no_show_rate
0,0-7 days,No,167,50,29.940120
1,0-7 days,Yes,473,128,27.061311
2,15-30 days,No,378,176,46.560847
3,15-30 days,Yes,955,400,41.884817
4,31-45 days,No,358,198,55.307263
5,31-45 days,Yes,900,479,53.222222
6,46-60 days,No,302,222,73.509934
7,46-60 days,Yes,865,568,65.664740
8,8-14 days,No,161,56,34.782609
9,8-14 days,Yes,441,146,33.106576


### 4.1 Order Booking Lead-Time Groups
The booking lead-time groups are arranged from the shortest to the longest lead time to make comparisons across reminder status easier to interpret.

In [19]:
lead_time_order = [
    "0-7 days",
    "8-14 days",
    "15-30 days",
    "31-45 days",
    "46-60 days"
]

lead_time_reminder["lead_time_group"] = pd.Categorical(
    lead_time_reminder["lead_time_group"],
    categories=lead_time_order,
    ordered=True
)

lead_time_reminder = lead_time_reminder.sort_values(
    ["lead_time_group", "reminder_sent"]
).reset_index(drop=True)

lead_time_reminder

,lead_time_group,reminder_sent,appointments,no_shows,no_show_rate
0,0-7 days,No,167,50,29.940120
1,0-7 days,Yes,473,128,27.061311
2,8-14 days,No,161,56,34.782609
3,8-14 days,Yes,441,146,33.106576
4,15-30 days,No,378,176,46.560847
5,15-30 days,Yes,955,400,41.884817
6,31-45 days,No,358,198,55.307263
7,31-45 days,Yes,900,479,53.222222
8,46-60 days,No,302,222,73.509934
9,46-60 days,Yes,865,568,65.664740


### 4.2 Review Appointment Counts
The number of appointments within each booking lead-time and reminder-status group will be reviewed before interpreting the no-show rates.
This helps provide context for the results by showing how many appointments are represented in each segment.

In [20]:
reminder_counts = pd.crosstab(
    df["lead_time_group"],
    df["reminder_sent"]
)

reminder_counts = reminder_counts.reindex(lead_time_order)

reminder_counts

reminder_sent,No,Yes
lead_time_group,,
0-7 days,167,473
8-14 days,161,441
15-30 days,378,955
31-45 days,358,900
46-60 days,302,865


### 4.3 Compare No-Show Rates
The no-show rates for appointments with and without reminders will be compared across each booking lead-time group.
This comparison will show whether the difference between reminder groups remains consistent as booking lead time increases.

In [21]:
reminder_rate_comparison = lead_time_reminder.pivot(
    index="lead_time_group",
    columns="reminder_sent",
    values="no_show_rate"
).round(2)

reminder_rate_comparison

reminder_sent,No,Yes
lead_time_group,,
0-7 days,29.94,27.06
8-14 days,34.78,33.11
15-30 days,46.56,41.88
31-45 days,55.31,53.22
46-60 days,73.51,65.66


### 4.4 Identify the Highest-Risk Reminder Segment
The combined results will be used to identify the lead-time and reminder-status segment with the highest no-show rate.
The appointment count for the identified segment will also be reviewed to provide context for the finding.

In [22]:
highest_reminder_risk = lead_time_reminder.loc[
    lead_time_reminder["no_show_rate"].idxmax()
]

print(f"Highest no-show rate: {highest_reminder_risk['no_show_rate']:.2f}%")
print(
    f"Segment: {highest_reminder_risk['lead_time_group']} "
    f"and reminder sent = {highest_reminder_risk['reminder_sent']}"
)
print(f"Appointments: {highest_reminder_risk['appointments']}")

Highest no-show rate: 73.51%
Segment: 46-60 days and reminder sent = No
Appointments: 302


In [23]:
reminder_rate_comparison

reminder_sent,No,Yes
lead_time_group,,
0-7 days,29.94,27.06
8-14 days,34.78,33.11
15-30 days,46.56,41.88
31-45 days,55.31,53.22
46-60 days,73.51,65.66


The results show that no-show rates generally increased as booking lead time increased, regardless of whether a reminder was sent.
For appointments with no reminder, the no-show rate increased from 29.94% for bookings made 0–7 days in advance to 73.51% for bookings made 46–60 days in advance. A similar pattern was observed among appointments where a reminder was sent, with the no-show rate increasing from 27.06% to 65.66%.
Appointments with reminders had lower no-show rates than appointments without reminders across all booking lead-time groups. However, the difference between the two groups remained relatively small compared with the overall increase associated with longer booking lead times.
The highest no-show rate was observed among appointments booked 46–60 days in advance without a reminder, at 73.51%, based on 302 appointments.
Overall, the findings suggest that longer booking lead time is associated with substantially higher no-show rates. Reminder status is also associated with lower no-show rates within each lead-time group, although reminders do not eliminate the higher no-show rates observed among appointments booked further in advance.

## 5. Advanced Analysis 3 — Previous No-Shows × Reminder Status
Week 5 showed that patients with previous no-shows had higher current no-show rates.
This analysis combines previous no-show history with reminder status to determine whether no-show rates differ depending on whether a reminder was sent.
The aim is to assess whether reminders are associated with lower no-show rates across different levels of previous no-show history and identify segments with particularly high no-show rates.

### 5.1 Calculate No-Show Rates by Previous No-Shows and Reminder Status
The appointments will be grouped by previous no-show history and reminder status.
For each combination, the number of appointments, number of no-shows, and no-show rate will be calculated. This will allow direct comparison of the two factors.

In [24]:
previous_no_show_reminder = (
    df.groupby(["previous_no_show_group", "reminder_sent"])
      .agg(
          appointments=("appointment_id", "count"),
          no_shows=("appointment_outcome", lambda x: (x == "No-Show").sum())
      )
      .reset_index()
)

previous_no_show_reminder["no_show_rate"] = (
    previous_no_show_reminder["no_shows"] /
    previous_no_show_reminder["appointments"] * 100
).round(2)

previous_no_show_reminder

,previous_no_show_group,reminder_sent,appointments,no_shows,no_show_rate
0,0,No,790,359,45.44
1,0,Yes,2131,912,42.80
2,1,No,428,242,56.54
3,1,Yes,1120,586,52.32
4,2+,No,148,101,68.24
5,2+,Yes,383,223,58.22


### 5.2 Compare No-Show Rates
The no-show rates will be compared across previous no-show history for appointments with and without reminders.
This comparison will help determine whether the difference associated with reminder status is consistent across the different previous no-show groups.

In [25]:
previous_no_show_reminder_comparison = previous_no_show_reminder.pivot(
    index="previous_no_show_group",
    columns="reminder_sent",
    values="no_show_rate"
).round(2)

previous_no_show_reminder_comparison

reminder_sent,No,Yes
previous_no_show_group,,
0,45.44,42.80
1,56.54,52.32
2+,68.24,58.22


### 5.3 Identify the Highest-Risk Segment
The combined results will be used to identify the previous no-show and reminder-status segment with the highest no-show rate.
The number of appointments in the identified segment will also be reviewed to provide context for the finding.

In [26]:
highest_previous_no_show_risk = previous_no_show_reminder.loc[
    previous_no_show_reminder["no_show_rate"].idxmax()
]

print(
    f"Highest no-show rate: "
    f"{highest_previous_no_show_risk['no_show_rate']:.2f}%"
)

print(
    f"Segment: {highest_previous_no_show_risk['previous_no_show_group']} "
    f"previous no-shows and reminder sent = "
    f"{highest_previous_no_show_risk['reminder_sent']}"
)

print(
    f"Appointments: "
    f"{highest_previous_no_show_risk['appointments']}"
)

Highest no-show rate: 68.24%
Segment: 2+ previous no-shows and reminder sent = No
Appointments: 148


The results show that no-show rates increased as previous no-show history increased, regardless of reminder status.
Among patients with no previous no-shows, the no-show rate was 45.44% when no reminder was sent compared with 42.80% when a reminder was sent. For patients with one previous no-show, the rates were 56.54% without a reminder and 52.32% with a reminder. Among patients with two or more previous no-shows, the no-show rate was 68.24% without a reminder compared with 58.22% when a reminder was sent.
The highest no-show rate was observed among patients with two or more previous no-shows who did not receive a reminder, at 68.24%, based on 148 appointments.
Overall, the findings suggest that previous no-show history is associated with higher current no-show rates. Reminder status was associated with lower no-show rates within each previous no-show group, with the largest difference observed among patients with two or more previous no-shows.

## 6. Advanced Analysis 4 — Waiting Time × Appointment Outcome

Waiting time was identified as a variable requiring further investigation after the earlier analysis.
This analysis examines whether the length of time patients wait at the clinic is associated with appointment outcomes.
The aim is to compare waiting-time patterns across attended and no-show appointments and determine whether longer waiting times are associated with higher no-show rates.

### 6.1 Review Waiting Time by Appointment Outcome
The distribution of waiting time will be reviewed across appointment outcomes to compare the typical waiting time for attended, no-show, and cancelled appointments.
Mean and median waiting times will be examined because the median provides an additional measure of the typical waiting time that is less affected by extreme values.

In [27]:
waiting_time_summary = (
    df.groupby("appointment_outcome")["waiting_time_minutes"]
      .agg(["count", "mean", "median"])
      .round(2)
)

waiting_time_summary

,count,mean,median
appointment_outcome,,,
Attended,2293,24.29,24.0
Cancelled,261,23.21,22.0
No-Show,2386,24.20,24.0


### 6.2 Group Waiting Time
To make the relationship between waiting time and appointment outcomes easier to compare, waiting times will be divided into meaningful time intervals.
The groups will allow no-show rates to be examined across increasing waiting-time ranges and help identify whether longer waiting periods are associated with higher no-show rates

In [28]:
waiting_bins = [0, 15, 30, 45, 60, float("inf")]
waiting_labels = [
    "0-15 minutes",
    "16-30 minutes",
    "31-45 minutes",
    "46-60 minutes",
    "60+ minutes"
]

df["waiting_time_group"] = pd.cut(
    df["waiting_time_minutes"],
    bins=waiting_bins,
    labels=waiting_labels,
    include_lowest=True
)

df["waiting_time_group"].value_counts().sort_index()

waiting_time_group
0-15 minutes     1073
16-30 minutes    2472
31-45 minutes    1260
46-60 minutes     132
60+ minutes         3
Name: count, dtype: int64

### 6.3 Calculate No-Show Rates by Waiting Time
The appointments will be grouped by waiting-time category to calculate the number of appointments, number of no-shows, and no-show rate for each waiting-time group.
This will help determine whether no-show rates change as waiting time increases.

In [29]:
waiting_time_analysis = (
    df.groupby("waiting_time_group", observed=False)
      .agg(
          appointments=("appointment_id", "count"),
          no_shows=("appointment_outcome", lambda x: (x == "No-Show").sum())
      )
      .reset_index()
)

waiting_time_analysis["no_show_rate"] = (
    waiting_time_analysis["no_shows"] /
    waiting_time_analysis["appointments"] * 100
).round(2)

waiting_time_analysis

,waiting_time_group,appointments,no_shows,no_show_rate
0,0-15 minutes,1073,505,47.06
1,16-30 minutes,2472,1216,49.19
2,31-45 minutes,1260,602,47.78
3,46-60 minutes,132,61,46.21
4,60+ minutes,3,2,66.67


### 6.4 Identify the Highest No-Show Waiting-Time Group
The waiting-time groups will be compared to identify the group with the highest no-show rate.
The number of appointments in the identified group will also be reviewed to provide context for the result.

In [30]:
highest_waiting_risk = waiting_time_analysis.loc[
    waiting_time_analysis["no_show_rate"].idxmax()
]

print(f"Highest no-show rate: {highest_waiting_risk['no_show_rate']:.2f}%")
print(f"Waiting-time group: {highest_waiting_risk['waiting_time_group']}")
print(f"Appointments: {highest_waiting_risk['appointments']}")

Highest no-show rate: 66.67%
Waiting-time group: 60+ minutes
Appointments: 3


The results do not show a clear or consistent relationship between waiting time and no-show rates.
Across the first four waiting-time groups, no-show rates remained relatively similar, ranging from 46.21% to 49.19%. The 60+ minutes group recorded the highest no-show rate at 66.67%, but this group contained only 3 appointments.
Due to the very small number of appointments in the 60+ minutes group, the result should be interpreted with caution and should not be considered sufficient evidence that very long waiting times are associated with higher no-show rates.
Overall, waiting time does not appear to be a strong or consistent predictor of no-shows in this dataset.

## 7. Advanced Analysis 5 — Distance to Clinic × Appointment Outcome
Distance to the clinic was identified as another variable requiring further investigation.
This analysis examines whether the distance patients travel to the clinic is associated with appointment outcomes.
The aim is to compare no-show rates across different distance ranges and determine whether patients living farther from the clinic show different attendance patterns.

### 7.1 Review Distance by Appointment Outcome
The distribution of distance to the clinic will be reviewed across appointment outcomes.
Mean and median distance will be compared for attended, no-show, and cancelled appointments to identify any differences in the typical distance travelled by patients.

In [31]:
distance_summary = (
    df.groupby("appointment_outcome")["distance_to_clinic_km"]
      .agg(["count", "mean", "median"])
      .round(2)
)

distance_summary

,count,mean,median
appointment_outcome,,,
Attended,2275,9.67,8.3
Cancelled,259,10.11,9.1
No-Show,2376,10.53,9.1


### 7.2 Group Distance to Clinic
To make the relationship between distance and appointment outcomes easier to compare, distance to the clinic will be divided into distance ranges.
The groups will allow no-show rates to be examined across increasing distances and help identify whether patients travelling farther to the clinic have different no-show patterns.

In [32]:
distance_bins = [0, 5, 10, 15, 20, float("inf")]
distance_labels = [
    "0-5 km",
    "6-10 km",
    "11-15 km",
    "16-20 km",
    "20+ km"
]

df["distance_group"] = pd.cut(
    df["distance_to_clinic_km"],
    bins=distance_bins,
    labels=distance_labels,
    include_lowest=True
)

df["distance_group"].value_counts().sort_index()

distance_group
0-5 km      1156
6-10 km     1692
11-15 km    1132
16-20 km     537
20+ km       393
Name: count, dtype: int64

### 7.3 Calculate No-Show Rates by Distance
The appointments will be grouped by distance category to calculate the number of appointments, number of no-shows, and no-show rate for each distance group.
This will help determine whether no-show rates change as the distance to the clinic increases.

In [33]:
distance_analysis = (
    df.groupby("distance_group", observed=False)
      .agg(
          appointments=("appointment_id", "count"),
          no_shows=("appointment_outcome", lambda x: (x == "No-Show").sum())
      )
      .reset_index()
)

distance_analysis["no_show_rate"] = (
    distance_analysis["no_shows"] /
    distance_analysis["appointments"] * 100
).round(2)

distance_analysis

,distance_group,appointments,no_shows,no_show_rate
0,0-5 km,1156,537,46.45
1,6-10 km,1692,787,46.51
2,11-15 km,1132,549,48.50
3,16-20 km,537,276,51.40
4,20+ km,393,227,57.76


### 7.4 Identify the Highest No-Show Distance Group
The distance groups will be compared to identify the group with the highest no-show rate.
The number of appointments in the identified group will also be reviewed to provide context for the result.

In [34]:
highest_distance_risk = distance_analysis.loc[
    distance_analysis["no_show_rate"].idxmax()
]

print(f"Highest no-show rate: {highest_distance_risk['no_show_rate']:.2f}%")
print(f"Distance group: {highest_distance_risk['distance_group']}")
print(f"Appointments: {highest_distance_risk['appointments']}")

Highest no-show rate: 57.76%
Distance group: 20+ km
Appointments: 393


The results show a general increase in no-show rates as distance to the clinic increases.
Patients living 0–5 km from the clinic had a no-show rate of 46.45%, while those travelling more than 20 km had the highest no-show rate at 57.76%. The no-show rate increased progressively across the longer-distance groups, reaching 51.40% among patients travelling 16–20 km.
The 20+ km group contained 393 appointments, providing a sufficient number of observations for the finding to be considered meaningful within this dataset.
Overall, the findings suggest that greater distance from the clinic is associated with higher no-show rates. Distance may therefore be a relevant factor to consider when identifying appointments that could benefit from additional attendance-support measures.

## 8. Advanced Analysis 6 — Age Group × Appointment Outcome
Age was identified as another variable requiring further investigation.
This analysis examines whether appointment outcomes differ across patient age groups.
The aim is to compare no-show rates across age groups and identify whether certain age groups have noticeably higher or lower no-show rates.

### 8.1 Calculate No-Show Rates by Age Group
The appointments will be grouped by age group to calculate the number of appointments, number of no-shows, and no-show rate for each age category.
This will allow comparison of no-show patterns across different age groups.

In [35]:
age_group_analysis = (
    df.groupby("age_group")
      .agg(
          appointments=("appointment_id", "count"),
          no_shows=("appointment_outcome", lambda x: (x == "No-Show").sum())
      )
      .reset_index()
)

age_group_analysis["no_show_rate"] = (
    age_group_analysis["no_shows"] /
    age_group_analysis["appointments"] * 100
).round(2)

age_group_analysis

,age_group,appointments,no_shows,no_show_rate
0,18-24,564,283,50.18
1,25-34,783,397,50.70
2,35-44,819,396,48.35
3,45-54,793,381,48.05
4,55-64,800,406,50.75
5,65+,1241,560,45.12


### 8.2 Order Age Groups
The age groups will be arranged in chronological order to make comparisons between younger and older patient groups easier to interpret.

In [36]:
age_group_order = [
    "18-24",
    "25-34",
    "35-44",
    "45-54",
    "55-64",
    "65+"
]

age_group_analysis["age_group"] = pd.Categorical(
    age_group_analysis["age_group"],
    categories=age_group_order,
    ordered=True
)

age_group_analysis = age_group_analysis.sort_values(
    "age_group"
).reset_index(drop=True)

age_group_analysis

,age_group,appointments,no_shows,no_show_rate
0,18-24,564,283,50.18
1,25-34,783,397,50.70
2,35-44,819,396,48.35
3,45-54,793,381,48.05
4,55-64,800,406,50.75
5,65+,1241,560,45.12


### 8.3 Identify Highest and Lowest No-Show Age Groups
The age groups will be compared to identify the groups with the highest and lowest no-show rates.
The number of appointments in each identified group will also be reviewed to provide context for the findings.

In [37]:
highest_age_risk = age_group_analysis.loc[
    age_group_analysis["no_show_rate"].idxmax()
]

lowest_age_risk = age_group_analysis.loc[
    age_group_analysis["no_show_rate"].idxmin()
]

print(f"Highest no-show rate: {highest_age_risk['no_show_rate']:.2f}%")
print(f"Age group: {highest_age_risk['age_group']}")
print(f"Appointments: {highest_age_risk['appointments']}")

print(f"\nLowest no-show rate: {lowest_age_risk['no_show_rate']:.2f}%")
print(f"Age group: {lowest_age_risk['age_group']}")
print(f"Appointments: {lowest_age_risk['appointments']}")

Highest no-show rate: 50.75%
Age group: 55-64
Appointments: 800

Lowest no-show rate: 45.12%
Age group: 65+
Appointments: 1241


No-show rates varied across the different age groups, but the differences were relatively small.
The 55–64 age group recorded the highest no-show rate at 50.75%, based on 800 appointments. The 65+ age group recorded the lowest no-show rate at 45.12%, based on 1,241 appointments.
The difference between the highest and lowest no-show rates was approximately 5.63 percentage points. This suggests that age group may have some association with appointment attendance, but it does not appear to be one of the strongest factors identified in the analysis.
The results indicate that age group can be considered as a supporting variable when understanding appointment attendance patterns, while greater attention may be given to stronger risk factors identified in the other Week 6 analyses.